# Pulse Metrics Analysis

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [ ]:
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import ShortTimeFFT, periodogram, welch
from scipy.signal.windows import hann

from rodeo import metrics

dbpath = Path("data/events/A03_2023-07-12/detections_SHRU905_VLA1.hdf5")

## Load database and extract time series

In [ ]:
count = 0
with h5py.File(dbpath, "r") as f:
    for chan in f.keys():
        print(chan)
        for k in f[chan].keys():
            det = f[chan][k]
            detection = metrics.build_detection(det)
            if count == 521:
                print("Breaking!")
                break
            count += 1
        break

x = detection.waveform.squeeze()
t = detection.time_vector
fs = detection.sampling_rate

## Plot time series & spectrogram

In [ ]:
plt.figure()
plt.plot(t, x)
plt.xlabel("Time")
plt.ylabel(f"Pressure [{detection.unit}]")
plt.show()

fmin = 10.0
fmax = 200.0

M = 2 ** 11
STFT = ShortTimeFFT(fs=fs, mfft=M, win=hann(M), hop=max(int(0.01 * M), 1), )
t_extent = STFT.extent(len(x))[:2]
extent = (t_extent[0], t_extent[1], fmin, fmax)
tvec = STFT.t(len(x))

Sxx = STFT.spectrogram(x)

find = np.argwhere((STFT.f >= fmin) & (STFT.f <= fmax))
tind = np.argwhere((tvec >= 0.0) & (tvec <= 1.0)).T

Sxx = Sxx[find, tind]
Sxx_dB = 10 * np.log10(Sxx)

plt.figure()
plt.imshow(Sxx_dB, aspect="auto", origin="lower", extent=extent, vmin=140, vmax=Sxx_dB.max())
plt.xlabel("Time [s]")
plt.ylabel("Frequency [Hz]")
# plt.axvline(t[STFT.k_min], color="r", linestyle="--")
# plt.yscale("log")
plt.show()

## Octaves

In [ ]:
start_freq = 10
num_freqs = 24
ratio = 2 ** (1/3)
values = np.array([start_freq, *(start_freq * np.cumprod(np.ones(num_freqs) * ratio))])
nfft = 2 ** 15

plt.figure(dpi=150, figsize=(6, 4))

f, Ppg = periodogram(x, fs=fs, window="hann", nfft=nfft)
Ppg = 10 * np.log10(Ppg)
plt.plot(f, Ppg, label="Periodogram")

f, Pw = welch(x, fs=fs, window="hann", nfft=nfft, nperseg=nfft / 4)
Pw = 10 * np.log10(Pw)
plt.plot(f, Pw, label="Welch's Method")

plt.xlim(10, 500)
plt.ylim(80, 140)
plt.grid("major")
plt.xlabel("Frequency [Hz]")
plt.ylabel("PSD [dB re 1 $\\mu$Pa/Hz]")
plt.legend()
plt.show()
print(len(f))

## Split spectrum into octaves

In [ ]:
bin_inds = np.digitize(f, values)
print(bin_inds.shape, f.shape)
print(np.unique(bin_inds))

octave_vals = np.bincount(bin_inds, weights=Pw) / np.bincount(bin_inds)
print(values.shape, octave_vals.shape)

plt.figure()
ax = plt.gca()
ax.plot(octave_vals[:-1], "-", label="Octave Bands")
ax.set_xticklabels(values[::3])
ax.grid("major")
# plt.plot(octave_vals)
plt.show()

print(values[::3])

## Sound pressure level (SPL)

$\mathrm{SPL}_i = 20 \log_{10} \left(\frac{\vert x_i \vert}{P_\mathrm{ref}}\right)$, where $P_\mathrm{ref} = 1\ \mathrm{\mu Pa}$

## Root-means-square (RMS) SPL:

$\mathrm{SPL}_\mathrm{RMS} = 20 \log_{10} \left( \sqrt{\frac{1}{N} \sum_{i=1}^N x_i^2} \right)$

## Zero-to-peak SPL:

$\mathrm{SPL}_\mathrm{ZP} = 20 \log_{10} \left( \max \vert x_i \vert \right)$

In [ ]:
x_spl = 20 * np.log10(np.abs(x))

plt.figure()
plt.plot(t, x_spl)
plt.xlabel("Time")
plt.ylabel(f"SPL [dB re 1 {detection.unit}]")
plt.show()

x_splrms = 20 * np.log10(np.sqrt(np.mean(x**2)))

print(f"RMS SPL = {x_splrms:.02f} dB re 1 {detection.unit}")

P_zp = np.abs(x).max()
SPL_zp = 20 * np.log10(P_zp)

print(f"Zero-to-Peak SPL = {SPL_zp:.02f} dB re 1 {detection.unit}")

## Sound exposure level (SEL):

$E_{90} = \int_{5}^{95} x^2(t) dt = \frac{1}{N_{95} - N_{5}}\sum_{i=N_5}^{N_{95}}x_i^2$

### Single-strike SEL:

$\mathrm{SEL}_\mathrm{SS} = 10 \log_{10} \left( \frac{E_{90}}{E_\mathrm{ref}} \right)$, where $E_\mathrm{ref} = 1\ \mu\mathrm{Pa}^2 \mathrm{s}$

In [ ]:
energy = x ** 2
cum_energy = np.cumsum(energy)
norm_cum_energy = cum_energy / cum_energy[-1]

lower_bound = 0.05
upper_bound = 0.95

start_index = np.searchsorted(norm_cum_energy, lower_bound)
end_index = np.searchsorted(norm_cum_energy, upper_bound)
print(f"Start index = {start_index}, End index = {end_index}")

E90 = 1 / (end_index - start_index) * np.sum(energy[start_index:end_index])

SEL_SS = 10 * np.log10(E90 / 1)
print(f"SEL_SS = {SEL_SS:.02f} dB re 1 {detection.unit}^2 s")